# 07 — BTXRD S0: SKELEX Image-Only

Mục tiêu: kiểm tra DenseNet-169 có đang là bottleneck hay không.

Checkpoint chính thức: `skhoha/SKELEX`.

Giữ nguyên:
- task 8 lớp;
- split 2510 / 538 / 538;
- class weights;
- Macro-F1 là metric chính;
- train-only augmentation nhẹ;
- val/test deterministic.

Fine-tuning SKELEX:
- AdamW;
- LR `5e-5`;
- weight decay `0.05`;
- layer-wise LR decay `0.75`;
- warmup 5 epochs;
- cosine schedule;
- 50 epochs;
- batch 16, gradient accumulation 2;
- chọn checkpoint theo validation Macro-F1.

Đây là pilot S0. Nếu SKELEX vượt rõ B0/B3, sau đó mới tích hợp nó vào GitHub codebase chính.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Paths và cấu hình

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Cao học/Experiments/BTXRD")
IMAGE_DIR = ROOT / "03_cache" / "images_512"
SPLITS_CSV = ROOT / "01_prep" / "splits.csv"
CLASS_WEIGHTS_CSV = ROOT / "01_prep" / "class_weights.csv"
RUN_DIR = ROOT / "04_runs" / "s0_skelex_image_8class"
RUN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "skhoha/SKELEX"

SEED = 2026
INPUT_SIZE = 224
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

EPOCHS = 50
BASE_LR = 5e-5
WEIGHT_DECAY = 0.05
LAYER_DECAY = 0.75
WARMUP_EPOCHS = 5
NUM_WORKERS = 2
USE_AMP = True
USE_PER_IMAGE_MINMAX = True

SMOKE_TEST = False
if SMOKE_TEST:
    EPOCHS = 2

print("RUN_DIR:", RUN_DIR)
print("MODEL:", MODEL_NAME)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)


## 3. Cài thư viện

In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.44", "scikit-learn>=1.2"],
    check=True,
)


## 4. Imports, seed, GPU

In [ ]:
import json, math, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import ViTForImageClassification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if DEVICE.type != "cuda":
    raise RuntimeError("SKELEX fine-tuning cần GPU.")
print("GPU:", torch.cuda.get_device_name(0))


## 5. Kiểm dữ liệu và class weights

In [ ]:
CLASS_NAMES = [
    "normal",
    "osteochondroma",
    "multiple_osteochondromas",
    "simple_bone_cyst",
    "giant_cell_tumor",
    "osteofibroma",
    "synovial_osteochondroma",
    "osteosarcoma",
]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i:c for c,i in CLASS_TO_IDX.items()}

assert IMAGE_DIR.exists()
assert SPLITS_CSV.exists()
assert CLASS_WEIGHTS_CSV.exists()

splits = pd.read_csv(SPLITS_CSV)
weights_df = pd.read_csv(CLASS_WEIGHTS_CSV)

assert len(splits) == 3586
assert splits["image_id"].is_unique
assert set(splits["class8"]) == set(CLASS_NAMES)
assert splits["split"].value_counts().to_dict() == {
    "train": 2510, "val": 538, "test": 538
}
assert len(list(IMAGE_DIR.glob("*.png"))) == 3586
assert weights_df["class8"].tolist() == CLASS_NAMES

class_weights_np = (
    weights_df.set_index("class8")
    .loc[CLASS_NAMES, "weight_sqrt_inv"]
    .to_numpy(dtype=np.float32)
)
class_weights = torch.tensor(
    class_weights_np, dtype=torch.float32, device=DEVICE
)

display(pd.crosstab(splits["class8"], splits["split"]))
display(weights_df)


## 6. Dataset và transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomAffine(
        degrees=7,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_skelex_rgb(path):
    with Image.open(path) as im:
        im = im.convert("L")
        arr = np.asarray(im).astype(np.float32)

    if USE_PER_IMAGE_MINMAX:
        lo = float(np.nanmin(arr))
        hi = float(np.nanmax(arr))
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            arr = (arr - lo) / (hi - lo) * 255.0
        else:
            arr = np.zeros_like(arr)

    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr).convert("RGB")

class BTXRDSkelexDataset(Dataset):
    def __init__(self, frame, transform):
        self.df = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        iid = row["image_id"]
        label = CLASS_TO_IDX[row["class8"]]
        path = IMAGE_DIR / f"{Path(iid).stem}.png"
        image = self.transform(load_skelex_rgb(path))
        return image, label, iid

train_df = splits[splits["split"]=="train"].reset_index(drop=True)
val_df = splits[splits["split"]=="val"].reset_index(drop=True)
test_df = splits[splits["split"]=="test"].reset_index(drop=True)

train_ds = BTXRDSkelexDataset(train_df, train_transform)
val_ds = BTXRDSkelexDataset(val_df, eval_transform)
test_ds = BTXRDSkelexDataset(test_df, eval_transform)

x, y, iid = train_ds[0]
print("Dataset sizes:", len(train_ds), len(val_ds), len(test_ds))
print("Example:", iid, x.shape, y)


## 7. DataLoaders

In [ ]:
train_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    generator=train_generator, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=NUM_WORKERS > 0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 8. Load SKELEX

In [ ]:
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,
)

model.config.id2label = {i: cls for i, cls in enumerate(CLASS_NAMES)}
model.config.label2id = {cls: i for i, cls in enumerate(CLASS_NAMES)}
model = model.to(DEVICE)

print("Hidden size:", model.config.hidden_size)
print("Layers:", model.config.num_hidden_layers)
print("Heads:", model.config.num_attention_heads)
print("Patch size:", model.config.patch_size)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_total:,}")
print(f"Trainable params: {n_trainable:,}")


## 9. AdamW + Layer-wise LR Decay

In [ ]:
def get_layer_id(name, num_layers):
    if name.startswith("vit.embeddings"):
        return 0
    if name.startswith("vit.encoder.layer."):
        return int(name.split(".")[3]) + 1
    if name.startswith("vit.layernorm"):
        return num_layers + 1
    return num_layers + 2

def build_llrd_groups(model, base_lr, weight_decay, layer_decay):
    num_layers = model.config.num_hidden_layers
    max_layer_id = num_layers + 2
    groups = {}

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        layer_id = get_layer_id(name, num_layers)
        lr = base_lr * (layer_decay ** (max_layer_id - layer_id))
        no_decay = (
            param.ndim == 1
            or name.endswith(".bias")
            or "layernorm" in name.lower()
        )
        wd = 0.0 if no_decay else weight_decay

        key = (layer_id, wd)
        if key not in groups:
            groups[key] = {
                "params": [],
                "lr": lr,
                "weight_decay": wd,
                "layer_id": layer_id,
            }
        groups[key]["params"].append(param)

    return list(groups.values())

param_groups = build_llrd_groups(
    model, BASE_LR, WEIGHT_DECAY, LAYER_DECAY
)
optimizer = torch.optim.AdamW(
    param_groups, lr=BASE_LR, betas=(0.9, 0.999)
)

print("Optimizer groups:", len(param_groups))
print("Min LR:", min(g["lr"] for g in param_groups))
print("Max LR:", max(g["lr"] for g in param_groups))


## 10. Warmup + cosine scheduler

In [ ]:
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = steps_per_epoch * WARMUP_EPOCHS

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lr_lambda
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler(
    "cuda", enabled=(USE_AMP and DEVICE.type=="cuda")
)

print("Steps/epoch:", steps_per_epoch)
print("Warmup steps:", warmup_steps)
print("Total steps:", total_steps)


## 11. Metrics

In [ ]:
def metrics_from_arrays(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    p, r, f1, support = precision_recall_fscore_support(
        y_true, y_pred,
        labels=list(range(len(CLASS_NAMES))),
        zero_division=0,
    )

    out = {
        "accuracy": float(acc),
        "balanced_accuracy": float(bacc),
        "macro_precision": float(np.mean(p)),
        "macro_recall": float(np.mean(r)),
        "macro_f1": float(np.mean(f1)),
    }

    if y_prob is not None:
        y_onehot = np.eye(len(CLASS_NAMES))[np.asarray(y_true)]
        try:
            out["macro_auroc_ovr"] = float(
                roc_auc_score(
                    y_onehot, np.asarray(y_prob),
                    average="macro", multi_class="ovr"
                )
            )
        except ValueError:
            out["macro_auroc_ovr"] = None

    for i, cls in enumerate(CLASS_NAMES):
        out[f"{cls}_precision"] = float(p[i])
        out[f"{cls}_recall"] = float(r[i])
        out[f"{cls}_f1"] = float(f1[i])
        out[f"{cls}_support"] = int(support[i])

    return out

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    n = 0
    ys, preds, probs, ids = [], [], [], []

    for images, labels, batch_ids in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            "cuda", enabled=(USE_AMP and DEVICE.type=="cuda")
        ):
            logits = model(pixel_values=images).logits
            loss = criterion(logits, labels)

        prob = torch.softmax(logits, dim=1)
        pred = prob.argmax(dim=1)

        bs = images.size(0)
        total_loss += loss.item() * bs
        n += bs

        ys.extend(labels.cpu().numpy().tolist())
        preds.extend(pred.cpu().numpy().tolist())
        probs.extend(prob.cpu().numpy().tolist())
        ids.extend(list(batch_ids))

    m = metrics_from_arrays(ys, preds, probs)
    m["loss"] = total_loss / n
    return m, np.asarray(ys), np.asarray(preds), np.asarray(probs), ids


## 12. Smoke forward/backward

In [ ]:
model.train()
images, labels, _ = next(iter(train_loader))
images = images.to(DEVICE)
labels = labels.to(DEVICE)

optimizer.zero_grad(set_to_none=True)

with torch.amp.autocast(
    "cuda", enabled=(USE_AMP and DEVICE.type=="cuda")
):
    logits = model(pixel_values=images).logits
    smoke_loss = criterion(logits, labels)

assert logits.shape == (images.size(0), 8)
assert torch.isfinite(smoke_loss)

scaler.scale(smoke_loss).backward()
optimizer.zero_grad(set_to_none=True)

print("Smoke logits:", logits.shape)
print("Smoke loss:", float(smoke_loss.detach().cpu()))
print("Forward/backward OK.")


## 13. Train S0 SKELEX full

In [ ]:
BEST_CKPT = RUN_DIR / "best.pt"
LAST_CKPT = RUN_DIR / "last.pt"
HISTORY_CSV = RUN_DIR / "history.csv"

history = []
best_macro_f1 = -1.0
best_epoch = -1
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    n_train = 0
    train_y, train_pred = [], []

    pbar = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"Epoch {epoch:02d}/{EPOCHS}",
        leave=False,
    )

    for batch_idx, (images, labels, _) in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            "cuda", enabled=(USE_AMP and DEVICE.type=="cuda")
        ):
            logits = model(pixel_values=images).logits
            raw_loss = criterion(logits, labels)
            loss = raw_loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if batch_idx % GRAD_ACCUM_STEPS == 0 or batch_idx == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        pred = logits.argmax(dim=1)
        bs = images.size(0)
        train_loss_sum += raw_loss.item() * bs
        n_train += bs
        train_y.extend(labels.detach().cpu().numpy().tolist())
        train_pred.extend(pred.detach().cpu().numpy().tolist())

        pbar.set_postfix(
            loss=f"{raw_loss.item():.4f}",
            lr=f"{optimizer.param_groups[-1]['lr']:.2e}",
        )

    train_metrics = metrics_from_arrays(train_y, train_pred)
    train_metrics["loss"] = train_loss_sum / n_train

    val_metrics, _, _, _, _ = evaluate(model, val_loader)

    row = {
        "epoch": epoch,
        "lr_head": optimizer.param_groups[-1]["lr"],
        **{f"train_{k}": v for k,v in train_metrics.items()},
        **{f"val_{k}": v for k,v in val_metrics.items()},
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    improved = val_metrics["macro_f1"] > best_macro_f1

    state = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_macro_f1": best_macro_f1,
        "best_epoch": best_epoch,
    }
    torch.save(state, LAST_CKPT)

    if improved:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        state["best_macro_f1"] = best_macro_f1
        state["best_epoch"] = best_epoch
        torch.save(state, BEST_CKPT)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss={train_metrics['loss']:.4f} "
        f"F1={train_metrics['macro_f1']:.4f} | "
        f"val loss={val_metrics['loss']:.4f} "
        f"F1={val_metrics['macro_f1']:.4f} "
        f"BAcc={val_metrics['balanced_accuracy']:.4f}"
        + ("  <-- best" if improved else "")
    )

elapsed = (time.time() - start_time) / 60
print("\nTraining complete")
print("Best epoch:", best_epoch)
print("Best val Macro-F1:", best_macro_f1)
print(f"Time: {elapsed:.1f} min")


## 14. Training curves

In [ ]:
history_df = pd.read_csv(HISTORY_CSV)

plt.figure(figsize=(8,5))
plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="Train Macro-F1")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="Val Macro-F1")
plt.axvline(best_epoch, linestyle="--", label=f"Best epoch {best_epoch}")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("S0 SKELEX")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 15. Evaluate best checkpoint trên test

In [ ]:
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])

test_metrics, y_true, y_pred, y_prob, test_ids = evaluate(
    model, test_loader
)

print("TEST METRICS")
for key in [
    "accuracy", "balanced_accuracy", "macro_precision",
    "macro_recall", "macro_f1", "macro_auroc_ovr", "loss"
]:
    value = test_metrics.get(key)
    if value is not None:
        print(f"{key:24s}: {value:.4f}")

print("\nClassification report")
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4, zero_division=0
))


## 16. Confusion matrix + per-class metrics

In [ ]:
cm = confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

fig, ax = plt.subplots(figsize=(11,9))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)
disp.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    xticks_rotation=45,
)
plt.title("S0 SKELEX — BTXRD 8-Class")
plt.tight_layout()
plt.show()

p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=list(range(len(CLASS_NAMES))),
    zero_division=0
)

per_class_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support,
})

display(per_class_df)


## 17. So sánh với B0 / B2 / B3

In [ ]:
comparison = pd.DataFrame([
    {
        "run": "B0 DenseNet image-only",
        "accuracy": 0.7454,
        "balanced_accuracy": 0.6035,
        "macro_f1": 0.6178,
        "macro_auroc": 0.9078,
    },
    {
        "run": "B2 DenseNet + knowledge",
        "accuracy": 0.7677,
        "balanced_accuracy": 0.5744,
        "macro_f1": 0.6133,
        "macro_auroc": 0.8965,
    },
    {
        "run": "B3 DenseNet + knowledge + contrastive",
        "accuracy": 0.7677,
        "balanced_accuracy": 0.5866,
        "macro_f1": 0.6264,
        "macro_auroc": 0.9131,
    },
    {
        "run": "S0 SKELEX image-only",
        "accuracy": test_metrics.get("accuracy"),
        "balanced_accuracy": test_metrics.get("balanced_accuracy"),
        "macro_f1": test_metrics.get("macro_f1"),
        "macro_auroc": test_metrics.get("macro_auroc_ovr"),
    },
])
display(comparison)


## 18. Save outputs

In [ ]:
pred_df = pd.DataFrame({
    "image_id": test_ids,
    "y_true": y_true,
    "y_pred": y_pred,
    "true_class": [IDX_TO_CLASS[i] for i in y_true],
    "pred_class": [IDX_TO_CLASS[i] for i in y_pred],
})
for i, cls in enumerate(CLASS_NAMES):
    pred_df[f"p_{cls}"] = y_prob[:, i]

pred_df.to_csv(RUN_DIR / "test_predictions.csv", index=False)
per_class_df.to_csv(RUN_DIR / "test_per_class_metrics.csv", index=False)

summary = {
    "run": "S0 SKELEX image-only",
    "model_name": MODEL_NAME,
    "best_epoch": int(best_epoch),
    "best_val_macro_f1": float(best_macro_f1),
    "test_metrics": test_metrics,
    "training": {
        "optimizer": "AdamW",
        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "layer_decay": LAYER_DECAY,
        "warmup_epochs": WARMUP_EPOCHS,
        "scheduler": "cosine",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "weighted_ce": True,
        "per_image_minmax": USE_PER_IMAGE_MINMAX,
    },
}

(RUN_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved outputs to:", RUN_DIR)


# Gửi lại sau khi chạy

Gửi:

```text
04_runs/s0_skelex_image_8class/
├── summary.json
├── history.csv
├── test_per_class_metrics.csv
└── test_predictions.csv
```

Nếu S0 vượt rõ B0/B3 về Macro-F1, bước sau là tích hợp SKELEX vào GitHub codebase chính và xây proposition model trên patch tokens của nó.
